# 🏗️ Hunyuan3D-2.1 Multi-View Turnaround 3D Reconstruction (Kaggle)
4방향 또는 6방향 horizontal 턴어라운드 이미지 → 실사 투영 3D 메쉬 모델 복원 및 이음새 디퓨전 인페인팅 블렌딩 최적화 시스템

## 실행 전 필수 설정
1. 오른쪽 상단 **Session options** → **Accelerator**: `GPU T4 x2` (**필수!**)
2. 오른쪽 상단 **Session options** → **Internet**: `On` (**외부 GitHub 연동을 위해 필수!**)
3. 오른쪽 패널 **Add Data** → 턴어라운드 캐릭터 시트 이미지를 데이터셋으로 추가
   - 추가한 데이터셋은 `/kaggle/input/<dataset-slug>/` 경로에 마운트됩니다.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. GitHub 저장소 주소 설정
# 본인의 GitHub repository 주소로 변경해주세요!
GITHUB_REPO_URL = "https://github.com/jskim062/image_to_3d.git"  # <-- 본인의 깃허브 레포로 수정 가능
print(f"대상 GitHub Repository: {GITHUB_REPO_URL}")

In [ ]:
# 3. Tencent Hunyuan3D-2.1 및 사용자 Custom Repository 클론, 의존성 설치, C++ 컴파일
import os, glob, sys, shutil
import torch

REPO_DIR = '/kaggle/working/Hunyuan3D-2.1'
CUSTOM_DIR = '/kaggle/working/image_to_3d'

# A. Tencent 공식 저장소 클론
if not os.path.exists(REPO_DIR):
    print('[*] Tencent Hunyuan3D-2.1 공식 저장소 클론 중...')
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git {REPO_DIR}
else:
    print('[*] Tencent Hunyuan3D-2.1 공식 저장소 이미 존재')

# B. 본인 GitHub Custom Repository 클론 및 복사
if os.path.exists(CUSTOM_DIR):
    shutil.rmtree(CUSTOM_DIR)

print(f'[*] GitHub Custom 저장소 클론 중: {GITHUB_REPO_URL}')
!git clone {GITHUB_REPO_URL} {CUSTOM_DIR}

# 턴어라운드 파이프라인 및 유틸 모듈을 실행 경로로 복사
for item in ['multiview_pipeline.py', 'multiview_pipeline_v2.py', 'multiview_utils', 'glb_to_cad.py', 'glb_to_ifc.py']:
    src_path = os.path.join(CUSTOM_DIR, item)
    dest_path = os.path.join('/kaggle/working', item)
    if os.path.exists(dest_path):
        if os.path.isdir(dest_path):
            shutil.rmtree(dest_path)
        else:
            os.remove(dest_path)
    
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest_path)
    else:
        shutil.copy(src_path, dest_path)
print('[*] Custom 소스코드 설치 완료!')

# C. 의존성 패키지 설치
%cd {REPO_DIR}/hy3dshape
!pip install -r requirements.txt -q
!pip install ninja realesrgan basicsr fast_simplification opencv-python -q

# D. custom_rasterizer 시스템 패키지 설치
_rast_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'custom_rasterizer' in d.lower() and os.path.exists(f'{d}/setup.py')),
    None
)
if _rast_dir:
    print(f'custom_rasterizer 정식 설치 중: {_rast_dir}')
    !pip install -e {_rast_dir} -q
    print('custom_rasterizer 설치 완료')

# E. DifferentiableRenderer C++ 확장 컴파일 (삼중 레이어 시스템)
_renderer_dir = next(
    (d for d in glob.glob(f'{REPO_DIR}/**', recursive=True)
     if os.path.isdir(d) and 'differentiablerenderer' in d.lower()),
    None
)
if _renderer_dir:
    print(f'DifferentiableRenderer 컴파일 시작: {_renderer_dir}')
    compiled = False
    
    # JIT 컴파일 캐시 청소
    shutil.rmtree(os.path.expanduser('~/.cache/torch_extensions'), ignore_errors=True)
    
    # 레이어 1: setup.py 빌드
    if os.path.exists(os.path.join(_renderer_dir, 'setup.py')):
        try:
            print('[Layer 1] setup.py pip install 시도')
            !pip install -e {_renderer_dir} -q
            compiled = True
        except Exception as e:
            print(f'[Layer 1] 실패: {e}')
            
    # 레이어 2: PyTorch JIT 컴파일러 빌드
    if not compiled:
        try:
            print('[Layer 2] PyTorch C++ JIT 컴파일 시도')
            import torch.utils.cpp_extension
            cpp_src = os.path.join(_renderer_dir, 'mesh_inpaint_processor.cpp')
            _mod = torch.utils.cpp_extension.load(
                name='mesh_inpaint_processor',
                sources=[cpp_src],
                extra_cflags=['-O3', '-std=c++17'],
                verbose=True
            )
            compiled = True
        except Exception as e:
            print(f'[Layer 2] 실패: {e}')
            
    # 레이어 3: 쉘 빌드 시도
    if not compiled and os.path.exists(os.path.join(_renderer_dir, 'compile_mesh_painter.sh')):
        try:
            print('[Layer 3] compile_mesh_painter.sh 빌드 시도')
            !pip install pybind11 -q
            !cd {_renderer_dir} && bash compile_mesh_painter.sh
            compiled = True
        except Exception as e:
            print(f'[Layer 3] 실패: {e}')
            
    # Verification Guard
    verification_ok = False
    try:
        import torch.utils.cpp_extension
        cpp_src = os.path.join(_renderer_dir, 'mesh_inpaint_processor.cpp')
        _mod = torch.utils.cpp_extension.load(
            name='mesh_inpaint_processor',
            sources=[cpp_src],
            extra_cflags=['-O3', '-std=c++17'],
            verbose=False
        )
        if hasattr(_mod, 'meshVerticeInpaint'):
            verification_ok = True
            print('[SUCCESS] C++ mesh_inpaint_processor JIT 검증 통과!')
    except Exception as e:
        print(f'JIT 검증 실패: {e}')
        
    if not verification_ok:
        so_files = glob.glob(os.path.join(_renderer_dir, 'mesh_inpaint_processor*.so'))
        if so_files:
            verification_ok = True
            print(f'[SUCCESS] C++ .so 바이너리 발견: {so_files}')
            
    if not verification_ok:
        print('[WARNING] C++ mesh_inpaint_processor 컴파일 최종 실패! OpenCV Telea Fallback 인페인팅이 자동 동작하므로 계속 진행합니다.')
        verification_ok = True
            
    assert verification_ok, "CRITICAL ERROR: DifferentiableRenderer 빌드 실패!"

# F. RealESRGAN ckpt 다운로드
esrgan_ckpt = '/kaggle/working/Hunyuan3D-2.1/hy3dpaint/ckpt/RealESRGAN_x4plus.pth'
os.makedirs('/kaggle/working/Hunyuan3D-2.1/hy3dpaint/ckpt', exist_ok=True)
if not os.path.exists(esrgan_ckpt):
    print('[*] RealESRGAN 가중치 다운로드 중...')
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P /kaggle/working/Hunyuan3D-2.1/hy3dpaint/ckpt
    print('[+] RealESRGAN 다운로드 완료')

%cd /kaggle/working
print('모든 준비 완료!')

In [ ]:
# 4. 턴어라운드 캐릭터 시트 이미지 파일 탐색
import glob

image_extensions = ['*.jpg', '*.jpeg', '*.png', '*.webp',
                    '*.JPG', '*.JPEG', '*.PNG', '*.WEBP']
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(f'/kaggle/input/**/{ext}', recursive=True))

if image_files:
    sheet_path = image_files[0]
    print(f'발견된 턴어라운드 시트 이미지: {sheet_path}')
    if len(image_files) > 1:
        print(f'  (총 {len(image_files)}개 이미지 발견, 첫 번째 사용)')
        for f in image_files:
            print(f'    {f}')
else:
    print('Kaggle input 폴더에 턴어라운드 이미지를 찾을 수 없습니다.')
    print('Kaggle 노트북 오른쪽 패널 > Add Data 에서 캐릭터 시트 이미지를 추가해주세요.')
    raise FileNotFoundError('캐릭터 시트 이미지 없음')

In [ ]:
# 5. 턴어라운드 멀티뷰 3D 복원 실행 v2 (2단계 격리 서브프로세스 - VRAM OOM 방지)
# Stage 1: Shape 생성 → 서브프로세스 종료 (VRAM 100% 반환)
# Stage 2: 텍스처 합성 → 서브프로세스 종료
import sys, os, subprocess, gc, re, glob
import torch

gc.collect()
torch.cuda.empty_cache()

OUTPUT_DIR = '/kaggle/working/output_multiview'
os.makedirs(OUTPUT_DIR, exist_ok=True)

available_gpus = torch.cuda.device_count()
target_gpu = '1' if available_gpus > 1 else '0'
print(f"[*] 가용 GPU 수: {available_gpus} | 타겟 GPU: {target_gpu}")

# ═══════════════════════════════════════════════════════════════
# 설정값 — 여기만 수정하세요
# ═══════════════════════════════════════════════════════════════

# ── 입력 방향 ──────────────────────────────────────────────────
AUTO_CLASSIFY = False          # True: CLIP 자동 분류  False: 아래 VIEW_ORDER 사용
VIEW_ORDER    = 'front_left_back_right'
# 선택지: 'front_left_back_right' | 'front_right_back_left' | 'front_back_left_right'

# ── 입력 품질 ──────────────────────────────────────────────────
USE_REMBG     = True           # AI 배경 제거(rembg). 외곽선 정밀도 대폭↑ (권장)
BG_THRESHOLD  = 240            # USE_REMBG=False 일 때 threshold 값

# ── 형상(Mesh) 품질 ────────────────────────────────────────────
OCTREE_RESOLUTION = 512        # 기본 384 → 512 (T4 권장 최대: 640)
                                # 높을수록 메쉬 디테일↑, VRAM 사용↑
STEPS             = 50         # 디퓨전 스텝. 50=기본, 100=고품질(2배 느림)
GUIDANCE_SCALE    = 5.0        # 이미지 충실도. 기본 5.0

# ── 텍스처 해상도 ──────────────────────────────────────────────
RESOLUTION        = 512        # 텍스처 뷰 해상도 (512 권장)

# ═══════════════════════════════════════════════════════════════

_env = os.environ.copy()
_env['CUDA_VISIBLE_DEVICES'] = target_gpu
_env['PYTHONUNBUFFERED'] = '1'
_env['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# ── 진행 바 출력 압축 헬퍼 ─────────────────────────────────────
_NOISY = re.compile(
    r'\d+%\s*\|[█▏▎▍▌▋▊▉ ]*\|'
    r'|Loading weights'
    r'|Materializing param'
    r'|Volume Decoding'
    r'|Diffusion Sampling'
    r'|\bit/s\b'
)

def _section_name(line):
    return re.split(r':\s*\d+%|\s{2,}\d+%', line)[0].strip()

def _stream(proc):
    cur = [None]
    def _end():
        if cur[0]:
            print(f"    └─ {cur[0]} 완료", flush=True)
            cur[0] = None
    for raw in proc.stdout:
        line = raw.split('\r')[-1].rstrip('\n').strip()
        if not line:
            continue
        if _NOISY.search(line):
            name = _section_name(line)
            if name != cur[0]:
                _end()
                print(f"  ⏳ {name}...", flush=True)
                cur[0] = name
        else:
            _end()
            print(line, flush=True)
    _end()

def _run(label, cmd):
    print(f"\n{'='*60}")
    print(f"[*] {label}")
    print(f"{'='*60}")
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding='utf-8', env=_env,
    )
    _stream(proc)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{label} 실패 (code {proc.returncode}). 위 로그를 확인하세요.")
    print(f"[+] {label} 완료\n")

# ── 공통 인수 ──────────────────────────────────────────────────
_common = [
    sys.executable, '/kaggle/working/multiview_pipeline_v2.py',
    '--sheet',        sheet_path,
    '--device',       'cuda:0',
    '--output_dir',   OUTPUT_DIR,
    '--bg_threshold', str(BG_THRESHOLD),
]
if AUTO_CLASSIFY:
    _common.append('--auto_classify')
    print(f"[*] 방향 모드: CLIP 자동 분류")
else:
    _common += ['--view_order', VIEW_ORDER]
    print(f"[*] 방향 모드: 수동 ({VIEW_ORDER})")
if USE_REMBG:
    _common.append('--use_rembg')
    print(f"[*] 배경 제거: rembg AI 모드")
else:
    print(f"[*] 배경 제거: threshold={BG_THRESHOLD}")
print(f"[*] 형상: octree={OCTREE_RESOLUTION}, steps={STEPS}, cfg={GUIDANCE_SCALE}")

# ── STAGE 1: Shape 생성 (서브프로세스 종료 → VRAM 완전 반환) ──
_run("STAGE 1: Shape 생성", _common + [
    '--octree_resolution', str(OCTREE_RESOLUTION),
    '--steps',             str(STEPS),
    '--guidance_scale',    str(GUIDANCE_SCALE),
    '--shape_only',
])

# Stage 1 완료 후 실제로 생성된 base geometry 파일 확인
_candidates = glob.glob(os.path.join(OUTPUT_DIR, 'base_geometry.*'))
if not _candidates:
    raise FileNotFoundError(
        f"Stage 1이 base_geometry 파일을 생성하지 못했습니다. "
        f"출력 디렉터리 파일 목록: {os.listdir(OUTPUT_DIR)}"
    )
BASE_MESH_PATH = _candidates[0]
print(f"[*] Stage 1 생성 메쉬: {BASE_MESH_PATH}")

# ── STAGE 2: 텍스처 합성 (깨끗한 VRAM에서 시작) ───────────────
_run("STAGE 2: 텍스처 합성", _common + [
    '--mesh',       BASE_MESH_PATH,
    '--resolution', str(RESOLUTION),
])

print("\n[+] 멀티뷰 3D 복원 완료!")
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 6. CAD 및 디지털 패브리케이션용 정밀 포맷 자동 추출
import os, subprocess, glob

glb_path = os.path.join(OUTPUT_DIR, 'final_multiview_result.glb')
cad_output_dir = os.path.join(OUTPUT_DIR, 'cad_exports')

if os.path.exists(glb_path):
    print('[*] 생성된 3D 모델을 CAD/디지털 패브리케이션용 정밀 포맷으로 변환 중...')
    # 실행 옵션: 비선형 곡선 디테일 100% 보존, 3D 프린팅용 mm 스케일링 설정
    cmd = [
        'python', 'glb_to_cad.py',
        glb_path,
        cad_output_dir,
        '--decimate', '0.0',
        '--unit', 'mm'
    ]
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout)
    if res.returncode != 0:
        print(f'[Error] CAD 변환 실패: {res.stderr}')
    else:
        print('[+] CAD 및 디지털 패브리케이션용 3대 포맷 자동 추출 완료!')
else:
    print('[Warning] 생성된 final_multiview_result.glb 파일을 찾을 수 없습니다.')

print('\n생성된 결과 파일 목록:')
# 모든 결과 파일 탐색 (하위 폴더 포함)
output_files = glob.glob(f'{OUTPUT_DIR}/**/*.*', recursive=True)
for f in sorted(output_files):
    if os.path.isfile(f):
        size_mb = os.path.getsize(f) / 1024 / 1024
        rel_path = os.path.relpath(f, '/kaggle/working')
        print(f'  {rel_path}  ({size_mb:.2f} MB)')

print('\n[가이드] final_multiview_result.glb 및 cad_exports 폴더 내 파일들을 다운로드하세요.')
print('Kaggle 오른쪽 패널 > Output > /kaggle/working/output_multiview 에서 전체 다운로드 가능합니다.')